# DEMForge Colab POC

Pipeline in this notebook:

1. Clone/install DEMForge.
2. Configure notebook-safe logging without force-resetting Colab logging.
3. Check runtime/GPU.
4. Optional synthetic smoke test.
5. Search/download USGS 1m DEM GeoTIFFs from TNMAccess.
6. Sanity-check rasters.
7. Build DEMForge training tiles.
8. Train the residual U-Net against real DEM tiles.
9. Sample/render predictions.

## Config

Run this cell first. Colab will show these as form controls / checkboxes.

Recommended first run:
- `USE_GOOGLE_DRIVE = True` if you want DEM downloads/checkpoints to survive runtime resets.
- `RUN_SYNTHETIC_SMOKE_TEST = False` once the project already passes smoke.
- `DOWNLOAD_LIMIT = 2` for the first real data run if Colab disk gets cranky.

In [ ]:
# @title DEMForge POC Config

# Core behavior toggles
USE_GOOGLE_DRIVE = True  # @param {type:"boolean"}
RUN_SYNTHETIC_SMOKE_TEST = False  # @param {type:"boolean"}
SEARCH_USGS_PRODUCTS = True  # @param {type:"boolean"}
DOWNLOAD_USGS_DEMS = True  # @param {type:"boolean"}
BUILD_REAL_DEM_TILES = True  # @param {type:"boolean"}
RUN_REAL_DEM_TRAINING = True  # @param {type:"boolean"}
SAMPLE_AFTER_TRAINING = True  # @param {type:"boolean"}
BACKUP_OUTPUTS_TO_DRIVE = True  # @param {type:"boolean"}

# Repo/project setup
REPO_URL = "https://github.com/Economy-Plus/demforge.git"  # @param {type:"string"}
PROJECT_DIR_STR = "/content/demforge"  # @param {type:"string"}
DRIVE_PROJECT_DIR_STR = "/content/drive/MyDrive/demforge_poc"  # @param {type:"string"}

# USGS AOI / product settings
AOI_NAME = "colorado_test"  # @param {type:"string"}
WEST = -106.55  # @param {type:"number"}
SOUTH = 39.55  # @param {type:"number"}
EAST = -106.25  # @param {type:"number"}
NORTH = 39.75  # @param {type:"number"}
MAX_PRODUCTS = 100  # @param {type:"integer"}
DOWNLOAD_LIMIT = 8  # @param {type:"integer"}

# Tile processing
TILE_SIZE = 512  # @param {type:"integer"}
STRIDE = 256  # @param {type:"integer"}
DOWNSCALE = 8  # @param {type:"integer"}

# Training / output
SAMPLE_COUNT = 8  # @param {type:"integer"}
SYNTHETIC_COUNT = 512  # @param {type:"integer"}
SYNTHETIC_SIZE = 256  # @param {type:"integer"}
FORCE_REDOWNLOAD = False  # @param {type:"boolean"}

from pathlib import Path

PROJECT_DIR = Path(PROJECT_DIR_STR)
DRIVE_PROJECT_DIR = Path(DRIVE_PROJECT_DIR_STR)

AOI = {
    "name": AOI_NAME,
    "west": float(WEST),
    "south": float(SOUTH),
    "east": float(EAST),
    "north": float(NORTH),
}

print("Config loaded:")
print(f"  USE_GOOGLE_DRIVE={USE_GOOGLE_DRIVE}")
print(f"  AOI={AOI}")
print(f"  DOWNLOAD_LIMIT={DOWNLOAD_LIMIT}")
print(f"  TILE_SIZE={TILE_SIZE}, STRIDE={STRIDE}, DOWNSCALE={DOWNSCALE}")

## Google Drive setup

This runs immediately after config so Drive auth happens early instead of ambushing you halfway through the pipeline.

In [ ]:
# @title Mount Google Drive early, if enabled

from pathlib import Path

DRIVE_MOUNTED = False

if USE_GOOGLE_DRIVE:
    from google.colab import drive

    print("Mounting Google Drive...")
    drive.mount("/content/drive")
    DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    DRIVE_MOUNTED = True

    print(f"Drive project directory: {DRIVE_PROJECT_DIR}")
else:
    print("Google Drive disabled. Using local Colab runtime storage only.")

# Raw DEMs are worth keeping on Drive if Drive is enabled.
RAW_ROOT = DRIVE_PROJECT_DIR / "data" / "raw" if DRIVE_MOUNTED else PROJECT_DIR / "data" / "raw"

# Processed tiles and active training outputs stay local for speed.
LOCAL_DATA_ROOT = PROJECT_DIR / "data"
LOCAL_OUTPUT_ROOT = PROJECT_DIR / "outputs"

print(f"RAW_ROOT={RAW_ROOT}")
print(f"LOCAL_DATA_ROOT={LOCAL_DATA_ROOT}")
print(f"LOCAL_OUTPUT_ROOT={LOCAL_OUTPUT_ROOT}")

## Setup: logging, clone, install

In [ ]:
# @title Setup: logging, clone, install

import os
import sys
import json
import time
import logging
import subprocess
from pathlib import Path

def get_logger(name: str = "demforge") -> logging.Logger:
    """Create a notebook-friendly logger without force-resetting global logging."""
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    logger.propagate = False

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s"
    )

    if not logger.handlers:
        handler = logging.StreamHandler(sys.stdout)
        handler.setFormatter(formatter)
        logger.addHandler(handler)
    else:
        for handler in logger.handlers:
            handler.setFormatter(formatter)

    return logger

logger = get_logger()
logger.info("Notebook logger ready.")

def run(cmd: str, cwd: Path | str | None = None, check: bool = True) -> subprocess.CompletedProcess:
    """Run a shell command with visible logging."""
    logger.info("$ %s", cmd)
    return subprocess.run(cmd, shell=True, cwd=str(cwd) if cwd else None, check=check)

logger.info("Checking for existing project directory...")
if not PROJECT_DIR.is_dir():
    logger.info("Cloning project from %s", REPO_URL)
    run(f"git clone {REPO_URL} {PROJECT_DIR}", cwd="/content")
else:
    logger.info("Project directory already exists: %s", PROJECT_DIR)

os.chdir(PROJECT_DIR)
logger.info("Working directory: %s", Path.cwd())

logger.info("Installing dependencies...")
run("python -m pip install --upgrade pip", check=False)
run("python -m pip install poetry", check=False)
run("python -m pip install -e .", check=True)
run("apt-get update && apt-get install -y inotify-tools rsync")

logger.info("Setup complete.")

## Runtime check

In [ ]:
import torch

logger.info("Python: %s", sys.version.replace("\n", " "))
logger.info("Torch: %s", torch.__version__)
logger.info("CUDA available: %s", torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = props.total_memory / 1024**3
    logger.info("GPU: %s", gpu_name)
    logger.info("VRAM: %.2f GB", vram_gb)
    logger.info("CUDA capability: %s", torch.cuda.get_device_capability(0))
    run("nvidia-smi", check=False)
else:
    logger.warning("No CUDA GPU detected. Dataset download/tiling is fine; training will be slow as hell.")

## Optional sanity test: synthetic dataset

In [ ]:
# @title Optional synthetic smoke test

if RUN_SYNTHETIC_SMOKE_TEST:
    logger.info("Building synthetic smoke-test dataset.")
    run(
        f"python scripts/make_synthetic_dataset.py "
        f"--out data/synthetic "
        f"--count {SYNTHETIC_COUNT} "
        f"--size {SYNTHETIC_SIZE}",
        check=True,
    )

    logger.info("Training smoke-test residual U-Net.")
    run("python scripts/train_residual_unet.py --config configs/residual_unet_smoke.yaml", check=True)
else:
    logger.info("Skipping synthetic smoke test.")

## USGS 1m DEM dataset search

In [ ]:
# @title USGS 1m DEM dataset search

import requests
from urllib.parse import urlparse
from tqdm.auto import tqdm

DATASET_NAME = "Digital Elevation Model (DEM) 1 meter"
PRODUCT_FORMAT = "GeoTIFF"

RAW_DIR = RAW_ROOT / "usgs_1m" / AOI["name"]
RAW_DIR.mkdir(parents=True, exist_ok=True)

def search_usgs_1m_dem_bbox(
    west: float,
    south: float,
    east: float,
    north: float,
    max_results: int = 100,
) -> list[dict]:
    """Search TNMAccess for USGS 1m DEM GeoTIFF products inside a bbox."""
    params = {
        "datasets": DATASET_NAME,
        "bbox": f"{west},{south},{east},{north}",
        "prodFormats": PRODUCT_FORMAT,
        "outputFormat": "JSON",
        "max": max_results,
    }

    response = requests.get(
        "https://tnmaccess.nationalmap.gov/api/v1/products",
        params=params,
        timeout=60,
    )
    response.raise_for_status()
    return response.json().get("items", [])

def get_download_url(item: dict) -> str | None:
    """Return the download URL from a TNMAccess product item."""
    return item.get("downloadURL") or item.get("downloadUrl")

def estimate_url_size(url: str) -> int | None:
    """Estimate remote file size using HTTP HEAD."""
    try:
        response = requests.head(url, allow_redirects=True, timeout=30)
        response.raise_for_status()
        size = response.headers.get("content-length")
        return int(size) if size else None
    except Exception as exc:
        logger.warning("Could not estimate size for %s: %s", url, exc)
        return None

def human_size(num_bytes: int | None) -> str:
    """Format bytes for display."""
    if num_bytes is None:
        return "unknown"

    units = ["B", "KB", "MB", "GB", "TB"]
    size = float(num_bytes)
    for unit in units:
        if size < 1024 or unit == units[-1]:
            return f"{size:.2f} {unit}"
        size /= 1024
    return f"{size:.2f} TB"

products = []

if SEARCH_USGS_PRODUCTS:
    logger.info("Searching TNMAccess for %s in AOI %s", DATASET_NAME, AOI["name"])

    items = search_usgs_1m_dem_bbox(
        west=AOI["west"],
        south=AOI["south"],
        east=AOI["east"],
        north=AOI["north"],
        max_results=MAX_PRODUCTS,
    )

    logger.info("Found %d candidate products.", len(items))

    known_total = 0

    for i, item in enumerate(items):
        url = get_download_url(item)
        size = estimate_url_size(url) if url else None
        if size is not None:
            known_total += size

        row = {
            "index": i,
            "title": item.get("title"),
            "url": url,
            "size_bytes": size,
            "size_human": human_size(size),
            "last_updated": item.get("lastUpdated") or item.get("modificationDate"),
        }
        products.append(row)

        logger.info("[%03d] %s | %s", i, row["size_human"], row["title"])
        logger.info("      %s", url)

    catalog_path = RAW_DIR / f"{AOI['name']}_catalog.json"
    catalog_path.write_text(json.dumps(products, indent=2), encoding="utf-8")

    logger.info("Known total size: %s", human_size(known_total))
    logger.info("Wrote catalog: %s", catalog_path)
else:
    catalog_path = RAW_DIR / f"{AOI['name']}_catalog.json"
    if catalog_path.exists():
        products = json.loads(catalog_path.read_text(encoding="utf-8"))
        logger.info("Loaded existing catalog: %s", catalog_path)
    else:
        raise RuntimeError("SEARCH_USGS_PRODUCTS is disabled, but no existing catalog was found.")

## Download selected DEM GeoTIFFs

In [ ]:
# @title Download selected DEM GeoTIFFs

def download_file(url: str, out_dir: Path, force: bool = False) -> Path:
    """Download one URL to out_dir with a progress bar."""
    out_dir.mkdir(parents=True, exist_ok=True)

    filename = Path(urlparse(url).path).name
    out_path = out_dir / filename

    if out_path.exists() and out_path.stat().st_size > 0 and not force:
        logger.info("Already exists: %s (%s)", out_path, human_size(out_path.stat().st_size))
        return out_path

    tmp_path = out_path.with_suffix(out_path.suffix + ".part")
    logger.info("Downloading: %s", url)

    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))

        with tmp_path.open("wb") as handle:
            with tqdm(total=total, unit="B", unit_scale=True, desc=filename) as bar:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        handle.write(chunk)
                        bar.update(len(chunk))

    tmp_path.rename(out_path)
    logger.info("Downloaded: %s (%s)", out_path, human_size(out_path.stat().st_size))
    return out_path

downloaded = []

if DOWNLOAD_USGS_DEMS:
    for product in products[:DOWNLOAD_LIMIT]:
        url = product["url"]
        if not url:
            logger.warning("Skipping product without URL: %s", product["title"])
            continue
        downloaded.append(download_file(url, RAW_DIR, force=FORCE_REDOWNLOAD))

    logger.info("Downloaded/available %d GeoTIFF files in %s", len(downloaded), RAW_DIR)

    urls_path = RAW_DIR / "downloaded_urls.txt"
    urls_path.write_text("\n".join(product["url"] for product in products[:DOWNLOAD_LIMIT] if product["url"]) + "\n", encoding="utf-8")
    logger.info("Wrote URL list: %s", urls_path)
else:
    logger.info("Skipping USGS DEM download. Existing files in RAW_DIR will be used.")

## Raster sanity check

In [ ]:
try:
    import rasterio
except Exception:
    logger.info("Installing rasterio.")
    run("python -m pip install rasterio", check=True)
    import rasterio

tifs = sorted(RAW_DIR.glob("*.tif")) + sorted(RAW_DIR.glob("*.tiff"))
logger.info("GeoTIFF count: %d", len(tifs))

if not tifs:
    raise RuntimeError(f"No GeoTIFFs found in {RAW_DIR}")

for tif in tifs[:5]:
    with rasterio.open(tif) as ds:
        logger.info("Raster: %s", tif.name)
        logger.info("  width x height: %s x %s", ds.width, ds.height)
        logger.info("  count: %s", ds.count)
        logger.info("  dtype: %s", ds.dtypes)
        logger.info("  crs: %s", ds.crs)
        logger.info("  bounds: %s", ds.bounds)
        logger.info("  nodata: %s", ds.nodata)

## Build DEMForge training tiles from downloaded DEMs

In [ ]:
# @title Build DEMForge training tiles from downloaded DEMs

import importlib
import inspect
import sys
from pathlib import Path
from time import perf_counter

counter = perf_counter()

TILES_DIR = LOCAL_DATA_ROOT / "tiles" / AOI["name"]

PROJECT_ROOT = Path("/content/demforge")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if BUILD_REAL_DEM_TILES:
    logger.info(
        "Building tiles: src=%s out=%s tile_size=%d stride=%d downscale=%d\n",
        RAW_DIR,
        TILES_DIR,
        TILE_SIZE,
        STRIDE,
        DOWNSCALE,
    )

    import scripts.build_tiles as build_tiles_module

    build_tiles_module = importlib.reload(build_tiles_module)

    kwargs = {
        "src": Path(RAW_DIR),
        "out": Path(TILES_DIR),
        "tile_size": TILE_SIZE,
        "stride": STRIDE,
        "downscale": DOWNSCALE,
        "val_fraction": 0.2,
        "min_valid_fraction": 0.98,
    }

    signature = inspect.signature(build_tiles_module.build_tiles)
    if "overwrite" in signature.parameters:
        kwargs["overwrite"] = False

    build_tiles_module.build_tiles(**kwargs)

else:
    logger.info("Skipping tile build. Existing tiles in %s will be used.", TILES_DIR)

train_count = len(list((TILES_DIR / "train").glob("*.npz")))
val_count = len(list((TILES_DIR / "val").glob("*.npz")))

logger.info("Train tiles: %d", train_count)
logger.info("Val tiles: %d", val_count)

if train_count == 0 or val_count == 0:
    raise RuntimeError(
        "Tile build produced an empty train or val split. "
        "Adjust AOI, download count, or build_tiles settings."
    )

logger.info("[DONE] Finished in %.2f seconds\n", perf_counter() - counter)

## Write a Colab training config for the real DEM dataset

In [ ]:
import yaml

def choose_training_profile() -> dict:
    """Choose a conservative config based on available VRAM."""
    if not torch.cuda.is_available():
        logger.warning("No CUDA; using tiny CPU config. This is mostly for plumbing, not serious training.")
        return {
            "base_channels": 16,
            "channel_mults": [1, 2, 2],
            "batch_size": 1,
            "grad_accum_steps": 4,
            "epochs": 2,
            "attention_at_bottleneck": False,
        }

    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    gpu_name = torch.cuda.get_device_name(0)

    if vram_gb >= 35:
        logger.info("Detected high-VRAM GPU profile: %s %.2fGB", gpu_name, vram_gb)
        return {
            "base_channels": 48,
            "channel_mults": [1, 2, 4, 4],
            "batch_size": 2,
            "grad_accum_steps": 8,
            "epochs": 20,
            "attention_at_bottleneck": True,
        }

    if vram_gb >= 20:
        logger.info("Detected mid/high GPU profile: %s %.2fGB", gpu_name, vram_gb)
        return {
            "base_channels": 48,
            "channel_mults": [1, 2, 4, 4],
            "batch_size": 2,
            "grad_accum_steps": 8,
            "epochs": 15,
            "attention_at_bottleneck": True,
        }

    logger.info("Detected conservative GPU profile: %s %.2fGB", gpu_name, vram_gb)
    return {
        "base_channels": 32,
        "channel_mults": [1, 2, 4],
        "batch_size": 1,
        "grad_accum_steps": 8,
        "epochs": 10,
        "attention_at_bottleneck": False,
    }

profile = choose_training_profile()

REAL_CONFIG = {
    "seed": 1337,
    "device": "auto",
    "data": {
        "train_dir": str(TILES_DIR / "train"),
        "val_dir": str(TILES_DIR / "val"),
        "num_workers": 2,
    },
    "model": {
        "in_channels": 4,
        "out_channels": 1,
        "base_channels": profile["base_channels"],
        "channel_mults": profile["channel_mults"],
        "attention_at_bottleneck": profile["attention_at_bottleneck"],
    },
    "train": {
        "epochs": profile["epochs"],
        "batch_size": profile["batch_size"],
        "grad_accum_steps": profile["grad_accum_steps"],
        "lr": 0.0002,
        "weight_decay": 0.00001,
        "amp": True,
        "clip_grad_norm": 1.0,
        "save_every_steps": 500,
        "render_every_steps": 500,
        "output_dir": str(LOCAL_OUTPUT_ROOT.relative_to(PROJECT_DIR) / f"usgs_1m_{AOI['name']}"),
    },
    "loss": {
        "height_l1": 1.0,
        "gradient_l1": 0.75,
        "laplacian_l1": 0.45,
        "spectral_l1": 0.15,
        "seam_l1": 0.20,
    },
}

config_path = PROJECT_DIR / "configs" / f"residual_unet_usgs_1m_{AOI['name']}.yaml"
config_path.write_text(yaml.safe_dump(REAL_CONFIG, sort_keys=False), encoding="utf-8")

logger.info("Wrote real DEM training config: %s", config_path)
print(config_path.read_text())

## Train against the real DEM dataset

In [ ]:
RUN_REAL_DEM_TRAINING = True

if RUN_REAL_DEM_TRAINING:
    logger.info("Starting real DEM training run.")
    run(f"python scripts/train_residual_unet.py --config {config_path}", check=True)
else:
    logger.info("Skipping real DEM training.")

## Sample and render predictions

In [ ]:
# @title Sample and render predictions

checkpoint = PROJECT_DIR / REAL_CONFIG["train"]["output_dir"] / "checkpoints" / "best.pt"
sample_out = PROJECT_DIR / REAL_CONFIG["train"]["output_dir"] / "samples"

if SAMPLE_AFTER_TRAINING:
    if not checkpoint.exists():
        raise FileNotFoundError(f"Expected checkpoint not found: {checkpoint}")

    logger.info("Sampling from checkpoint: %s", checkpoint)

    run(
        f"python scripts/sample.py "
        f"--checkpoint {checkpoint} "
        f"--data {TILES_DIR / 'val'} "
        f"--out {sample_out} "
        f"--count {SAMPLE_COUNT}",
        check=True,
    )

    logger.info("Sample outputs written to: %s", sample_out)
    run(f"find {sample_out} -maxdepth 1 -type f | sort | head -40", check=False)
else:
    logger.info("Skipping sampling/rendering.")

## Optional: save outputs to Google Drive

In [ ]:
# @title Optional: save outputs to Google Drive

if USE_GOOGLE_DRIVE and BACKUP_OUTPUTS_TO_DRIVE:
    drive_out = DRIVE_PROJECT_DIR / "outputs" / AOI["name"]
    drive_out.mkdir(parents=True, exist_ok=True)

    logger.info("Copying outputs to Drive: %s", drive_out)
    run(f"cp -r {PROJECT_DIR / REAL_CONFIG['train']['output_dir']} {drive_out}/", check=True)
    run(f"cp {config_path} {drive_out}/", check=True)
else:
    logger.info("Skipping Drive backup.")